In [ ]:
# 05_factor — CELL 1: how many independent risk factors are there in rural India?
import pandas as pd, numpy as np, statsmodels.api as sm
from google.colab import drive; drive.mount('/content/drive')
BASE = '/content/drive/My Drive/Data/Elevar'

crops = pd.read_parquet(f'{BASE}/crops_clean.parquet')
r     = pd.read_parquet(f'{BASE}/rainfall_shock.parquet'); r = r[r['Dist Code']<800]
P = crops.merge(r, on=['Dist Code','Year'], how='inner').dropna(subset=['ln_ag_output'])

# ---- shock = residual after district mean + district-specific quadratic trend ----
res = []
for dc, d in P.groupby('Dist Code'):
    d = d.sort_values('Year')
    if len(d) < 40: continue
    t = d.Year.values - d.Year.values.mean()
    X = sm.add_constant(np.column_stack([t, t**2]))
    f = sm.OLS(d.ln_ag_output.values, X).fit()
    res.append(pd.DataFrame({'Dist Code':dc,'Year':d.Year.values,'shock':f.resid}))
R = pd.concat(res)
W = R.pivot(index='Year', columns='Dist Code', values='shock')
W = W.loc[:, W.notna().mean() >= 0.95].dropna()          # balanced block
print(f'balanced matrix: {W.shape[0]} years x {W.shape[1]} districts')
print(f'years {W.index.min()}-{W.index.max()} | mean shock SD {W.std().mean():.3f}')

# ---- correlation structure ----
C = W.corr()
off = C.values[np.triu_indices_from(C, k=1)]
print(f'\nPAIRWISE CORRELATION of district output shocks')
print(f'  mean {off.mean():+.3f} | median {np.median(off):+.3f} | share positive {(off>0).mean():.1%}')
print(f'  if districts were independent, mean would be ~0')

# ---- eigenvalue spectrum ----
ev = np.linalg.eigvalsh(C)[::-1]
ve = ev/ev.sum()
print(f'\nFACTOR STRUCTURE ({C.shape[0]} districts)')
for i in range(8):
    print(f'  PC{i+1}: {ve[i]:6.1%}   cumulative {ve[:i+1].sum():6.1%}')
print(f'  PCs to reach 50% of variance: {np.argmax(np.cumsum(ve)>=0.50)+1}')
print(f'  PCs to reach 80% of variance: {np.argmax(np.cumsum(ve)>=0.80)+1}')

# participation ratio: effective number of independent factors
n_eff_factors = (ev.sum()**2)/(ev**2).sum()
print(f'\n  EFFECTIVE NUMBER OF FACTORS = {n_eff_factors:.1f}  (out of {C.shape[0]} districts)')

# ---- the portfolio number ----
print('\n=== EFFECTIVE DIVERSIFICATION ===')
print('an equal-weighted portfolio of k districts: how much does risk actually fall?')
rng = np.random.default_rng(7); cols = list(W.columns)
rows=[]
for k in [1,2,5,10,20,40,80,150]:
    if k > len(cols): break
    sds=[]
    for _ in range(400):
        pick = rng.choice(cols, size=k, replace=False)
        sds.append(W[pick].mean(axis=1).std())
    sd = np.mean(sds); sd1 = np.mean([W[c].std() for c in cols])
    # if independent, sd would fall as sd1/sqrt(k). solve for the k that WOULD give this sd
    k_eff = (sd1/sd)**2
    rows.append({'k districts':k,'portfolio SD':round(sd,4),
                 'if independent':round(sd1/np.sqrt(k),4),
                 'EFFECTIVE k':round(k_eff,1),
                 'diversification lost':f'{1-k_eff/k:.0%}'})
D = pd.DataFrame(rows); print(D.to_string(index=False))

kmax = D['k districts'].max()
print(f"\n  HEADLINE: a lender in {kmax} districts is effectively diversified across "
      f"{D[D['k districts']==kmax]['EFFECTIVE k'].iloc[0]:.0f}.")

# ---- does the first factor look like the monsoon? ----
from numpy.linalg import eigh
vals, vecs = eigh(C.values)
pc1 = W.values @ vecs[:, -1]
nat_rain = r.groupby('Year').rain_z.mean().reindex(W.index)
print(f'\nSANITY: corr(PC1 score, national mean rainfall z) = {np.corrcoef(pc1, nat_rain.fillna(0))[0,1]:+.3f}')
print('  strong |r| => the dominant factor IS the all-India monsoon')

W.to_parquet(f'{BASE}/shock_matrix.parquet')
pd.Series(ve[:50]).to_csv(f'{BASE}/eigen_spectrum.csv')
print('\nsaved shock_matrix.parquet, eigen_spectrum.csv')

Mounted at /content/drive
balanced matrix: 38 years x 214 districts
years 1970-2015 | mean shock SD 0.258

PAIRWISE CORRELATION of district output shocks
  mean +0.117 | median +0.102 | share positive 69.1%
  if districts were independent, mean would be ~0

FACTOR STRUCTURE (214 districts)
  PC1:  16.9%   cumulative  16.9%
  PC2:  10.0%   cumulative  27.0%
  PC3:   8.8%   cumulative  35.8%
  PC4:   6.7%   cumulative  42.5%
  PC5:   6.0%   cumulative  48.5%
  PC6:   5.0%   cumulative  53.5%
  PC7:   4.3%   cumulative  57.8%
  PC8:   3.9%   cumulative  61.7%
  PCs to reach 50% of variance: 6
  PCs to reach 80% of variance: 16

  EFFECTIVE NUMBER OF FACTORS = 14.8  (out of 214 districts)

=== EFFECTIVE DIVERSIFICATION ===
an equal-weighted portfolio of k districts: how much does risk actually fall?
 k districts  portfolio SD  if independent  EFFECTIVE k diversification lost
           1        0.2521          0.2580          1.0                  -5%
           2        0.1947          0.1

In [ ]:
# 05_factor — CELL 2: name the risk regions
import pandas as pd, numpy as np
from sklearn.decomposition import PCA

!pip install factor_analyzer
from factor_analyzer.rotator import Rotator
BASE = '/content/drive/My Drive/Data/Elevar'

W = pd.read_parquet(f'{BASE}/shock_matrix.parquet')
meta = (pd.read_parquet(f'{BASE}/crops_clean.parquet')
          .groupby('Dist Code')[['Dist Name','State Name']].first())

K = 6
Z = (W - W.mean())/W.std()
p = PCA(n_components=K).fit(Z.values)
load = p.components_.T * np.sqrt(p.explained_variance_)      # loadings
try:
    load = Rotator(method='varimax').fit_transform(load)      # varimax for interpretability
    print('varimax rotation applied')
except Exception:
    print('rotation unavailable, using unrotated loadings')

L = pd.DataFrame(load, index=W.columns, columns=[f'F{i+1}' for i in range(K)]).join(meta)
print(f'\nvariance explained by {K} factors: {p.explained_variance_ratio_.sum():.1%}')

print('\n=== FACTOR LOADINGS BY STATE (mean) ===')
st = L.groupby('State Name')[[f'F{i+1}' for i in range(K)]].mean().round(2)
st['n'] = L.groupby('State Name').size()
print(st.sort_values('F1', ascending=False).to_string())

print('\n=== WHAT EACH FACTOR IS: top-loading districts ===')
for i in range(K):
    f = f'F{i+1}'
    top = L.reindex(L[f].abs().sort_values(ascending=False).index).head(8)
    states = top['State Name'].value_counts().head(3).index.tolist()
    print(f'\n{f}  ({p.explained_variance_ratio_[i]:.1%})  -> {", ".join(states)}')
    print('   ' + ' | '.join(f'{r["Dist Name"]} {r[f]:+.2f}' for _, r in top.iterrows()))

# assign each district to its dominant factor = its risk region
L['region'] = L[[f'F{i+1}' for i in range(K)]].abs().idxmax(axis=1)
print('\n=== DISTRICTS PER RISK REGION ===')
print(pd.crosstab(L.region, L['State Name']).T.loc[
      lambda d: d.sum(axis=1).sort_values(ascending=False).index].head(20).to_string())

L.reset_index().to_parquet(f'{BASE}/factor_loadings.parquet', index=False)
print('\nsaved factor_loadings.parquet')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for factor_analyzer: filename=factor_analyzer-0.5.1-py2.py3-none-any.whl size=42655 sha256=3c8ca9904c91fc8d0e69c66252869b1b0501b0276c0a567038ba70ec53caa5b4
  Stored in directory: /root/.cache/pip/wheels/87/6e/8f/914f20e0242ee0214e5c8336031a2fab12e632e4695fbb7276
Successfully built factor_analyzer
varimax rotation applied

variance explained by 6 factors: 53.5%

=== FACTOR LOADINGS BY STATE (mean) ===
                    F1    F2    F3    F4    F5    F6   n
State Name                                              
Madhya Pradesh    0.57  0.08 -0.25  0.09 -0.03 -0.02  32
Assam             0.43 -0.11 -0.26  0.14  0.07 -0.15   3
Uttar Pradesh     0.33  0.15 -0.05  0.51 -0.13 -0.05  27
Himachal Pradesh  0.30  0.44  0.22 -0.49 -0.27 -0.18   9
Orissa            0.26 

In [ ]:
# 05_factor — CELL 3: portfolio overlay — what is a real footprint worth?
import pandas as pd, numpy as np
BASE = '/content/drive/My Drive/Data/Elevar'
W = pd.read_parquet(f'{BASE}/shock_matrix.parquet')
L = pd.read_parquet(f'{BASE}/factor_loadings.parquet').set_index('Dist Code')
sd1 = W.std().mean()

def eff_k(cols):
    cols = [c for c in cols if c in W.columns]
    if len(cols) < 2: return len(cols), np.nan, np.nan
    sd = W[cols].mean(axis=1).std()
    return len(cols), sd, (sd1/sd)**2

def report(name, cols):
    n, sd, k = eff_k(cols)
    print(f'{name:34s} n={n:3d}  SD={sd:.4f}  effective k={k:4.1f}  '
          f'({k/n:.0%} of nominal)' if n>=2 else f'{name}: too few districts')

# ---------- 1. concentrated vs spread, same size ----------
print('=== SAME SIZE, DIFFERENT SPREAD (n=20) ===')
rng = np.random.default_rng(11)
# worst case: all from one state
for s in ['Rajasthan','Maharashtra','Uttar Pradesh','Madhya Pradesh']:
    cols = L[L['State Name']==s].index.tolist()[:20]
    if len(cols)>=8: report(f'  concentrated: {s}', cols)
# best case: spread across risk regions
regions = L.region.unique()
spread = []
for i in range(20):
    reg = regions[i % len(regions)]
    pool = [c for c in L[L.region==reg].index if c in W.columns and c not in spread]
    if pool: spread.append(rng.choice(pool))
report('  spread across risk regions', spread)
# random 20
rand = rng.choice([c for c in W.columns], 20, replace=False)
report('  random 20 districts', list(rand))

# ---------- 2. the frontier: best achievable at each size ----------
print('\n=== BEST ACHIEVABLE DIVERSIFICATION BY PORTFOLIO SIZE ===')
print(f"{'n':>4} {'random k_eff':>13} {'region-spread k_eff':>21} {'gain':>8}")
for n in [5,10,20,40,80]:
    rk=[]; sk=[]
    for _ in range(200):
        rk.append(eff_k(list(rng.choice(list(W.columns), n, replace=False)))[2])
        sel=[]
        for i in range(n):
            reg = regions[i % len(regions)]
            pool = [c for c in L[L.region==reg].index if c in W.columns and c not in sel]
            if pool: sel.append(rng.choice(pool))
        sk.append(eff_k(sel)[2])
    print(f'{n:>4} {np.nanmean(rk):>13.1f} {np.nanmean(sk):>21.1f} {np.nanmean(sk)/np.nanmean(rk)-1:>7.0%}')

# ---------- 3. worked example: an Elevar-style footprint ----------
# EDIT THIS LIST to the districts Elevar's portfolio companies actually operate in
ELEVAR = ['Ahmadnagar','Solapur','Osmanabad','Beed','Latur','Nasik','Pune','Satara',
          'Jaipur','Ajmer','Jodhpur','Nagaur','Bhilwara','Udaipur',
          'Chittoor','Anantapur','Kurnool','Cuddapah','Nellore']
cols = L[L['Dist Name'].isin(ELEVAR)].index.tolist()
print('\n=== WORKED EXAMPLE: illustrative footprint ===')
n, sd, k = eff_k(cols)
print(f'  districts matched: {n} of {len(ELEVAR)} named')
print(f'  effective k = {k:.1f}  -> you hold {n} districts, you are diversified across {k:.0f}')
print(f'  region mix:'); print('   ', L.loc[cols].region.value_counts().to_dict())
best = []
for i in range(n):
    reg = regions[i % len(regions)]
    pool = [c for c in L[L.region==reg].index if c in W.columns and c not in best]
    if pool: best.append(rng.choice(pool))
print(f'  best achievable at n={n}: effective k = {eff_k(best)[2]:.1f}')
print(f'  RECOMMENDATION HEADROOM: {eff_k(best)[2]-k:+.1f} effective districts from re-spreading alone')

=== SAME SIZE, DIFFERENT SPREAD (n=20) ===
  concentrated: Rajasthan          n= 20  SD=0.3826  effective k= 0.5  (2% of nominal)
  concentrated: Maharashtra        n= 20  SD=0.1598  effective k= 2.6  (13% of nominal)
  concentrated: Uttar Pradesh      n= 20  SD=0.1144  effective k= 5.1  (25% of nominal)
  concentrated: Madhya Pradesh     n= 20  SD=0.1939  effective k= 1.8  (9% of nominal)
  spread across risk regions       n= 20  SD=0.0991  effective k= 6.8  (34% of nominal)
  random 20 districts              n= 20  SD=0.1460  effective k= 3.1  (16% of nominal)

=== BEST ACHIEVABLE DIVERSIFICATION BY PORTFOLIO SIZE ===
   n  random k_eff   region-spread k_eff     gain
   5           4.3                   4.7      7%
  10           5.0                   5.9     16%
  20           6.3                   8.0     26%
  40           6.1                   7.7     26%
  80           6.6                   8.6     29%

=== WORKED EXAMPLE: illustrative footprint ===
  districts matched: 14 of 19

In [ ]:
# 05_factor — CELL 4: CureBay footprint
import pandas as pd, numpy as np, re

!pip install rapidfuzz
from rapidfuzz import process, fuzz
BASE = '/content/drive/My Drive/Data/Elevar'
W = pd.read_parquet(f'{BASE}/shock_matrix.parquet')
L = pd.read_parquet(f'{BASE}/factor_loadings.parquet').set_index('Dist Code')
sd1 = W.std().mean()

CUREBAY = ['Angul','Sambalpur','Khordha','Nayagarh','Ganjam','Baudhgarh','Puri','Balod','Bemetara']

def norm(s):
    s = re.split(r'[/(]', str(s).lower())[0]
    return re.sub(r'[^a-z]','',s)
pool = L.assign(key=L['Dist Name'].map(norm))
hits=[]
for d in CUREBAY:
    m = process.extractOne(norm(d), pool.key.tolist(), scorer=fuzz.WRatio)
    if m and m[1] >= 78:
        r = pool.iloc[m[2]]
        hits.append({'named':d,'icrisat':r['Dist Name'],'state':r['State Name'],
                     'region':r['region'],'score':round(m[1],1),'code':pool.index[m[2]]})
H = pd.DataFrame(hits).drop_duplicates('code')
print(H.to_string(index=False))

cols = [c for c in H.code if c in W.columns]
if len(cols) >= 2:
    sd = W[cols].mean(axis=1).std(); k = (sd1/sd)**2
    print(f'\nCureBay: {len(cols)} ICRISAT parents | SD {sd:.4f} | EFFECTIVE k = {k:.1f}')
    print('region mix:', H[H.code.isin(cols)].region.value_counts().to_dict())
    rng = np.random.default_rng(3); regions = L.region.unique(); best=[]
    for i in range(len(cols)):
        reg = regions[i % len(regions)]
        p = [c for c in L[L.region==reg].index if c in W.columns and c not in best]
        if p: best.append(rng.choice(p))
    bsd = W[best].mean(axis=1).std()
    print(f'best achievable at n={len(cols)}: k = {(sd1/bsd)**2:.1f}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.8 MB/s eta 0:00:00
    named   icrisat  state region  score  code
Sambalpur Sambalpur Orissa     F1  100.0   712
   Ganjam    Ganjam Orissa     F2  100.0   705
     Puri      Puri Orissa     F5  100.0   711

CureBay: 3 ICRISAT parents | SD 0.2040 | EFFECTIVE k = 1.6
region mix: {'F1': 1, 'F2': 1, 'F5': 1}
best achievable at n=3: k = 2.1


In [ ]:
# 05_factor — CELL 5: the sector's effective diversification
import pandas as pd, numpy as np, glob
BASE = '/content/drive/My Drive/Data/Elevar'
W  = pd.read_parquet(f'{BASE}/shock_matrix.parquet')
L  = pd.read_parquet(f'{BASE}/factor_loadings.parquet').set_index('Dist Code')
BR = pd.read_csv(f'{BASE}/pc11_to_icrisat_bridge.csv')
sd1 = W.std().mean()

rbi = pd.read_csv(glob.glob(f'{BASE}/shrug/**/rbi_directory_shrid.csv', recursive=True)[0], low_memory=False)
print('branches:', len(rbi), '| groups:', rbi.rbi_bank_group.value_counts().head(8).to_dict())

rbi = rbi.rename(columns={'pc11_state_id':'pc11_state_id','pc11_district_id':'pc11_district_id'})
B = (rbi.groupby(['pc11_state_id','pc11_district_id']).size().rename('branches').reset_index()
        .merge(BR[['pc11_state_id','pc11_district_id','dist_code']], on=['pc11_state_id','pc11_district_id'])
        .groupby('dist_code').branches.sum())

def eff_k(cols, wts=None):
    cols = [c for c in cols if c in W.columns]
    if len(cols) < 2: return np.nan, np.nan
    if wts is None: s = W[cols].mean(axis=1)
    else:
        w = np.array([wts[c] for c in cols]); w = w/w.sum()
        s = (W[cols] * w).sum(axis=1)
    sd = s.std(); return sd, (sd1/sd)**2

# --- rural-weighted: RRB + small finance + cooperative branches proxy the rural book ---
RUR = rbi[rbi.rbi_bank_group.astype(str).str.contains('Regional Rural|Small Finance|Co-operative|Cooperative', case=False, na=False)]
Bx = (RUR.groupby(['pc11_state_id','pc11_district_id']).size().rename('b').reset_index()
         .merge(BR[['pc11_state_id','pc11_district_id','dist_code']], on=['pc11_state_id','pc11_district_id'])
         .groupby('dist_code').b.sum())
print(f'\nrural-facing branches: {RUR.shape[0]:,} in {Bx.shape[0]} ICRISAT parents')

for lab, series in [('ALL banks', B), ('RURAL-facing banks', Bx)]:
    cols = [c for c in series.index if c in W.columns]
    wts  = series.to_dict()
    sd_e, k_e = eff_k(cols)                      # equal-weighted
    sd_w, k_w = eff_k(cols, wts)                 # branch-weighted = the real book
    print(f'\n{lab}: {len(cols)} districts')
    print(f'   equal-weighted   effective k = {k_e:5.1f}')
    print(f'   branch-weighted  effective k = {k_w:5.1f}   <- the sector\'s ACTUAL exposure')
    top = series.reindex(cols).nlargest(30)
    sdt, kt = eff_k(top.index.tolist(), wts)
    print(f'   top-30 districts by branches: effective k = {kt:.1f}')
    reg = L.reindex(cols).assign(b=series.reindex(cols)).groupby('region').b.sum()
    print('   branch share by risk region:', (reg/reg.sum()).round(3).to_dict())

branches: 154505 | groups: {'NATIONALISED BANKS': 69166, 'PRIVATE SECTOR BANKS': 34475, 'SBI AND ITS ASSOCIATES': 23383, 'REGIONAL RURAL BANKS': 22249, 'SMALL FINANCE BANK': 4000, 'PAYMENTS BANK': 818, 'FOREIGN BANKS': 319, 'LOCAL AREA BANKS': 95}

rural-facing branches: 26,249 in 305 ICRISAT parents

ALL banks: 212 districts
   equal-weighted   effective k =   6.4
   branch-weighted  effective k =  10.4   <- the sector's ACTUAL exposure
   top-30 districts by branches: effective k = 17.4
   branch share by risk region: {'F1': 0.132, 'F2': 0.194, 'F3': 0.194, 'F4': 0.198, 'F5': 0.159, 'F6': 0.123}

RURAL-facing banks: 212 districts
   equal-weighted   effective k =   6.4
   branch-weighted  effective k =   8.0   <- the sector's ACTUAL exposure
   top-30 districts by branches: effective k = 9.6
   branch share by risk region: {'F1': 0.147, 'F2': 0.22, 'F3': 0.164, 'F4': 0.218, 'F5': 0.141, 'F6': 0.11}


In [ ]:
# 05_factor — CELL 6: decompose k_eff into volatility selection vs correlation
import pandas as pd, numpy as np, glob
BASE = '/content/drive/My Drive/Data/Elevar'
W  = pd.read_parquet(f'{BASE}/shock_matrix.parquet')
L  = pd.read_parquet(f'{BASE}/factor_loadings.parquet').set_index('Dist Code')
BR = pd.read_csv(f'{BASE}/pc11_to_icrisat_bridge.csv')
sd_glob = W.std().mean()

rbi = pd.read_csv(glob.glob(f'{BASE}/shrug/**/rbi_directory_shrid.csv', recursive=True)[0], low_memory=False)
def to_parent(df):
    return (df.groupby(['pc11_state_id','pc11_district_id']).size().rename('b').reset_index()
              .merge(BR[['pc11_state_id','pc11_district_id','dist_code']], on=['pc11_state_id','pc11_district_id'])
              .groupby('dist_code').b.sum())
ALL = to_parent(rbi)
RUR = to_parent(rbi[rbi.rbi_bank_group.astype(str).str.contains('Regional Rural|Small Finance|Co-operative|Cooperative', case=False, na=False)])

def decompose(cols, wts=None, label=''):
    cols = [c for c in cols if c in W.columns]
    if wts is None: w = np.ones(len(cols))/len(cols)
    else:
        w = np.array([wts.get(c,0) for c in cols], float); w = w/w.sum()
    port_sd = (W[cols]*w).sum(axis=1).std()
    ind_sd  = float((W[cols].std().values * w).sum())      # weighted mean individual SD
    k_pure  = (ind_sd/port_sd)**2                           # pure correlation effect
    k_naive = (sd_glob/port_sd)**2                          # what we reported before
    vol_sel = (sd_glob/ind_sd)**2                           # volatility-selection multiplier
    print(f'{label:34s} n={len(cols):3d} | indiv SD {ind_sd:.3f} vs global {sd_glob:.3f}')
    print(f'{"":34s}   k_pure(correlation) {k_pure:5.1f} | vol-selection x{vol_sel:4.2f} | naive {k_naive:5.1f}')
    return k_pure, vol_sel

print('=== DECOMPOSED ===')
decompose(list(W.columns), None, 'all 212, equal weight')
decompose(ALL.index.tolist(), ALL.to_dict(), 'all banks, branch-weighted')
decompose(RUR.index.tolist(), RUR.to_dict(), 'rural banks, branch-weighted')
decompose(ALL.reindex([c for c in ALL.index if c in W.columns]).nlargest(30).index.tolist(),
          ALL.to_dict(), 'top-30 branch districts')
for s in ['Rajasthan','Maharashtra','Uttar Pradesh','Madhya Pradesh']:
    decompose(L[L['State Name']==s].index.tolist()[:20], None, f'concentrated: {s}')
rng = np.random.default_rng(5); regions = L.region.unique(); spread=[]
for i in range(20):
    p = [c for c in L[L.region==regions[i%len(regions)]].index if c in W.columns and c not in spread]
    if p: spread.append(rng.choice(p))
decompose(spread, None, 'spread across 6 regions')

=== DECOMPOSED ===
all 212, equal weight              n=214 | indiv SD 0.258 vs global 0.258
                                     k_pure(correlation)   6.5 | vol-selection x1.00 | naive   6.5
all banks, branch-weighted         n=212 | indiv SD 0.220 vs global 0.258
                                     k_pure(correlation)   7.6 | vol-selection x1.38 | naive  10.4
rural banks, branch-weighted       n=212 | indiv SD 0.238 vs global 0.258
                                     k_pure(correlation)   6.9 | vol-selection x1.17 | naive   8.0
top-30 branch districts            n= 30 | indiv SD 0.180 vs global 0.258
                                     k_pure(correlation)   8.4 | vol-selection x2.06 | naive  17.4
concentrated: Rajasthan            n= 20 | indiv SD 0.498 vs global 0.258
                                     k_pure(correlation)   1.7 | vol-selection x0.27 | naive   0.5
concentrated: Maharashtra          n= 20 | indiv SD 0.232 vs global 0.258
                                     k_pur

(7.038894653449737, np.float64(2.1213753805772457))

In [ ]:
# 05_factor — CELL 7: the three tests
import pandas as pd, numpy as np, statsmodels.api as sm
BASE = '/content/drive/My Drive/Data/Elevar'
W = pd.read_parquet(f'{BASE}/shock_matrix.parquet')
L = pd.read_parquet(f'{BASE}/factor_loadings.parquet').set_index('Dist Code')
F = pd.read_parquet(f'{BASE}/features_union.parquet').set_index('Dist Code')
r = pd.read_parquet(f'{BASE}/rainfall_shock.parquet'); r = r[r['Dist Code']<800]

# ---------- national monsoon factor ----------
nat = r.groupby('Year').rain_z.mean().reindex(W.index)
Z = (W - W.mean())/W.std()
pc1 = pd.Series(np.linalg.eigh(Z.corr().values)[1][:,-1] @ Z.T.values, index=W.index)
if np.corrcoef(pc1, nat.fillna(0))[0,1] < 0: pc1 = -pc1
print(f'PC1 vs national rainfall: {np.corrcoef(pc1, nat.fillna(0))[0,1]:+.3f}')

# ---------- TEST 1: systematic vs idiosyncratic share per district ----------
rows=[]
for c in W.columns:
    m = sm.OLS(W[c].values, sm.add_constant(pc1.values)).fit()
    rows.append({'Dist Code':c,'beta':m.params[1],'r2_sys':m.rsquared,
                 'idio_sd':np.std(m.resid),'tot_sd':W[c].std()})
S = pd.DataFrame(rows).set_index('Dist Code').join(F, how='left').join(L[['region']], how='left')
print(f'\nSYSTEMATIC SHARE across {len(S)} districts: mean R2 {S.r2_sys.mean():.3f}, '
      f'range {S.r2_sys.min():.2f}-{S.r2_sys.max():.2f}')

CAND = ['nonfarm_sh','hhind_sh','ec_emp_pc','pop_density','pc11_td_city_dist',
        'irr_share','canal_share','tubewell_share','secc_unirr_sh','literacy2',
        'agworkers_pc','kharif_normal','rain_cv','secc_cons_pc_rural']
CAND = [c for c in CAND if c in S.columns and S[c].notna().mean()>0.7]
print('\n*** corr with SYSTEMATIC SHARE (r2_sys) — negative = makes risk idiosyncratic ***')
print(S[CAND+['r2_sys']].corr()['r2_sys'].drop('r2_sys').sort_values().round(3).to_string())
print('\n(for contrast) corr with TOTAL exposure magnitude:')
print(S[CAND+['tot_sd']].corr()['tot_sd'].drop('tot_sd').sort_values().round(3).to_string())

X = S[CAND].apply(lambda s: (s-s.mean())/s.std()).fillna(0)
m = sm.OLS(S.r2_sys.values, sm.add_constant(X)).fit(cov_type='HC3')
print(f'\nr2_sys ~ structure (standardised, N {len(S)}, R2 {m.rsquared:.3f})')
o = pd.DataFrame({'coef':m.params,'t':m.tvalues,'p':m.pvalues}).drop('const')
print(o.reindex(o.t.abs().sort_values(ascending=False).index).round(3).to_string())

# ---------- TEST 2: tail dependence ----------
print('\n*** TAIL DEPENDENCE: do districts co-move more in bad years? ***')
bad  = nat[nat < nat.quantile(0.25)].index
good = nat[nat > nat.quantile(0.75)].index
mid  = nat[(nat>=nat.quantile(0.25)) & (nat<=nat.quantile(0.75))].index
sd1 = W.std().mean()
for lab, yrs in [('DROUGHT years (bottom 25%)',bad), ('normal years',mid), ('GOOD years (top 25%)',good)]:
    sub = W.loc[yrs]
    C = sub.corr(); off = C.values[np.triu_indices_from(C,k=1)]
    isd = sub.std().mean(); psd = sub.mean(axis=1).std()
    print(f'  {lab:28s} n={len(yrs):2d} | mean corr {np.nanmean(off):+.3f} | k_pure {(isd/psd)**2:5.1f}')

# ---------- TEST 3: is correlation rising? ----------
print('\n*** IS DIVERSIFICATION DETERIORATING? 12-year rolling ***')
yrs = sorted(W.index)
out=[]
for i in range(0, len(yrs)-11, 3):
    win = yrs[i:i+12]; sub = W.loc[win]
    C = sub.corr(); off = C.values[np.triu_indices_from(C,k=1)]
    out.append({'window':f'{win[0]}-{win[-1]}','mean_corr':np.nanmean(off),
                'k_pure':(sub.std().mean()/sub.mean(axis=1).std())**2})
T = pd.DataFrame(out); print(T.round(3).to_string(index=False))
sl = sm.OLS(T.mean_corr.values, sm.add_constant(np.arange(len(T)))).fit()
print(f'  trend in mean correlation: {sl.params[1]:+.4f} per window (t {sl.tvalues[1]:.1f})')

S.reset_index().to_parquet(f'{BASE}/systematic_share.parquet', index=False)

PC1 vs national rainfall: +0.733

SYSTEMATIC SHARE across 214 districts: mean R2 0.169, range 0.00-0.54

*** corr with SYSTEMATIC SHARE (r2_sys) — negative = makes risk idiosyncratic ***
secc_cons_pc_rural   -0.464
literacy2            -0.441
ec_emp_pc            -0.316
nonfarm_sh           -0.227
hhind_sh             -0.014
canal_share          -0.003
secc_unirr_sh         0.011
kharif_normal         0.025
pop_density           0.058
agworkers_pc          0.082
pc11_td_city_dist     0.089
irr_share             0.175
tubewell_share        0.189
rain_cv               0.204

(for contrast) corr with TOTAL exposure magnitude:
pop_density          -0.390
irr_share            -0.380
kharif_normal        -0.273
nonfarm_sh           -0.199
literacy2            -0.194
hhind_sh             -0.192
tubewell_share       -0.158
ec_emp_pc            -0.134
canal_share          -0.095
secc_cons_pc_rural   -0.037
agworkers_pc          0.228
pc11_td_city_dist     0.328
rain_cv               0.335
secc_

In [ ]:
# 05_factor — CELL 8: does the prosperity finding survive region fixed effects?
import pandas as pd, numpy as np, statsmodels.api as sm
BASE = '/content/drive/My Drive/Data/Elevar'
S = pd.read_parquet(f'{BASE}/systematic_share.parquet').set_index('Dist Code')

CORE = ['secc_cons_pc_rural','literacy2','irr_share','tubewell_share',
        'nonfarm_sh','ec_emp_pc','rain_cv','kharif_normal','pop_density']
CORE = [c for c in CORE if c in S.columns]
D = S.dropna(subset=['r2_sys','region']+CORE).copy()
X = D[CORE].apply(lambda s:(s-s.mean())/s.std())
print(f'N {len(D)}')

def run(X, lab, groups=None):
    m = sm.OLS(D.r2_sys.values, sm.add_constant(X)).fit(
        cov_type='cluster', cov_kwds={'groups':groups}) if groups is not None \
        else sm.OLS(D.r2_sys.values, sm.add_constant(X)).fit(cov_type='HC3')
    print(f'\n{lab}  R2 {m.rsquared:.3f}')
    o = pd.DataFrame({'coef':m.params,'t':m.tvalues,'p':m.pvalues}).drop('const', errors='ignore')
    o = o[~o.index.str.startswith('reg_')]
    print(o.reindex(o.t.abs().sort_values(ascending=False).index).round(3).to_string())

run(X, 'NO CONTROLS')
Xr = pd.concat([X, pd.get_dummies(D.region, prefix='reg', drop_first=True, dtype=float)], axis=1)
run(Xr, 'WITH RISK-REGION FIXED EFFECTS', groups=D.region.values)
Xs = pd.concat([X, pd.get_dummies(D['State Name'], prefix='reg', drop_first=True, dtype=float)], axis=1)
run(Xs, 'WITH STATE FIXED EFFECTS', groups=D['State Name'].values)

# placebo: does prosperity predict the LEVEL of risk? (should stay null)
m = sm.OLS(D.tot_sd.values, sm.add_constant(Xr)).fit(cov_type='cluster', cov_kwds={'groups':D.region.values})
print(f'\nPLACEBO — same model on TOTAL exposure (R2 {m.rsquared:.3f}):')
print(f'  secc_cons_pc_rural {m.params["secc_cons_pc_rural"]:+.4f} (t {m.tvalues["secc_cons_pc_rural"]:.2f})')
print(f'  literacy2          {m.params["literacy2"]:+.4f} (t {m.tvalues["literacy2"]:.2f})')
print('  want: near zero -> prosperity changes the CHARACTER of risk, not its SIZE')

# the money chart: systematic share by consumption quintile
D['q'] = pd.qcut(D.secc_cons_pc_rural, 5, labels=['Q1 poorest','Q2','Q3','Q4','Q5 richest'])
print('\n=== SYSTEMATIC SHARE BY DISTRICT PROSPERITY ===')
print(D.groupby('q', observed=True).agg(n=('r2_sys','size'), systematic_share=('r2_sys','mean'),
        total_exposure=('tot_sd','mean'), idiosyncratic_sd=('idio_sd','mean')).round(3).to_string())

N 203

NO CONTROLS  R2 0.366
                     coef      t      p
secc_cons_pc_rural -0.070 -5.502  0.000
rain_cv             0.036  3.332  0.001
literacy2          -0.042 -3.296  0.001
kharif_normal       0.030  2.740  0.006
nonfarm_sh          0.018  1.378  0.168
irr_share           0.011  0.923  0.356
ec_emp_pc           0.012  0.923  0.356
tubewell_share      0.005  0.379  0.704
pop_density         0.005  0.332  0.740

WITH RISK-REGION FIXED EFFECTS  R2 0.556
                     coef      t      p
literacy2          -0.023 -5.337  0.000
secc_cons_pc_rural -0.047 -4.885  0.000
pop_density         0.017  2.536  0.011
kharif_normal       0.013  1.789  0.074
nonfarm_sh          0.015  1.572  0.116
ec_emp_pc          -0.005 -1.458  0.145
irr_share           0.011  1.025  0.305
rain_cv             0.007  0.596  0.551
tubewell_share      0.004  0.592  0.554

WITH STATE FIXED EFFECTS  R2 0.547
                     coef      t      p
secc_cons_pc_rural -0.036 -2.042  0.041
literacy2    

In [ ]:
# 05_factor — CELL 9: beta decomposition + mechanism
import pandas as pd, numpy as np, statsmodels.api as sm
BASE = '/content/drive/My Drive/Data/Elevar'
S = pd.read_parquet(f'{BASE}/systematic_share.parquet').set_index('Dist Code')

# ---------- A. is it beta, or is it extra noise? ----------
S['sys_sd'] = np.sqrt((S.tot_sd**2 - S.idio_sd**2).clip(lower=0))
D = S.dropna(subset=['r2_sys','region','secc_cons_pc_rural']).copy()
D['q'] = pd.qcut(D.secc_cons_pc_rural, 5, labels=['Q1 poorest','Q2','Q3','Q4','Q5 richest'])
print('=== IS IT BETA OR NOISE? ===')
print(D.groupby('q', observed=True)[['beta','sys_sd','idio_sd','tot_sd','r2_sys']].mean().round(3).to_string())
print('\n  if beta and sys_sd FALL while idio_sd is FLAT -> prosperity cuts the cord, not adds noise')
for y in ['beta','sys_sd','idio_sd']:
    x = (D.secc_cons_pc_rural-D.secc_cons_pc_rural.mean())/D.secc_cons_pc_rural.std()
    Xr = pd.concat([x.rename('cons'), pd.get_dummies(D.region, prefix='r', drop_first=True, dtype=float)], axis=1)
    m = sm.OLS(D[y].values, sm.add_constant(Xr)).fit(cov_type='cluster', cov_kwds={'groups':D.region.values})
    print(f'  {y:8s} ~ prosperity (region FE): {m.params["cons"]:+.4f} (t {m.tvalues["cons"]:.2f})')

# ---------- B. bring in horticulture, never used ----------
raw = pd.read_csv(f'{BASE}/ICRISAT-District Level Data.csv', low_memory=False)
raw['Year'] = pd.to_numeric(raw['Year'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
v = [c for c in raw.columns if 'AREA' in c or 'PRODUCTION' in c]
raw[v] = raw[v].apply(pd.to_numeric, errors='coerce').mask(lambda x: x<0)
H = ['FRUITS AND VEGETABLES AREA (1000 ha)','POTATOES AREA (1000 ha)','ONION AREA (1000 ha)']
H = [c for c in H if c in raw.columns]
rec = raw[raw.Year.between(2000,2015)]
hort = rec.groupby('Dist Code')[H].mean().sum(axis=1).rename('hort_area')
gca  = rec.groupby('Dist Code')[[c for c in raw.columns if c.endswith('AREA (1000 ha)')]].mean().sum(axis=1)
D['hort_share'] = (hort/gca).reindex(D.index)
print(f'\nhort_share: mean {D.hort_share.mean():.3f}, coverage {D.hort_share.notna().mean():.0%}')
print(f'  corr(hort_share, r2_sys)      = {D.hort_share.corr(D.r2_sys):+.3f}')
print(f'  corr(hort_share, prosperity)  = {D.hort_share.corr(D.secc_cons_pc_rural):+.3f}')

# ---------- C. MEDIATION: what kills the prosperity coefficient? ----------
MED = {'baseline'          : [],
       '+ horticulture'    : ['hort_share'],
       '+ non-farm work'   : ['nonfarm_sh','ec_emp_pc'],
       '+ irrigation'      : ['irr_share','canal_share','tubewell_share'],
       '+ crop mix'        : ['msp_cereal_share','coarse_cereal_share','oilseeds_share','cash_crop_share'],
       '+ market access'   : ['pc11_td_city_dist','pop_density'],
       '+ climate'         : ['kharif_normal','rain_cv'],
       '+ EVERYTHING'      : ['hort_share','nonfarm_sh','ec_emp_pc','irr_share','canal_share',
                              'tubewell_share','msp_cereal_share','coarse_cereal_share',
                              'oilseeds_share','cash_crop_share','pc11_td_city_dist',
                              'pop_density','kharif_normal','rain_cv']}
print('\n=== MEDIATION: does the prosperity effect survive each channel? ===')
print(f"{'model':20s} {'coef':>8s} {'t':>7s} {'% of baseline':>15s}  R2")
base=None
for lab, meds in MED.items():
    meds = [m for m in meds if m in D.columns]
    cols = ['secc_cons_pc_rural','literacy2'] + meds
    Q = D.dropna(subset=cols).copy()
    X = Q[cols].apply(lambda s:(s-s.mean())/s.std())
    X = pd.concat([X, pd.get_dummies(Q.region, prefix='r', drop_first=True, dtype=float)], axis=1)
    m = sm.OLS(Q.r2_sys.values, sm.add_constant(X)).fit(cov_type='cluster', cov_kwds={'groups':Q.region.values})
    c, t = m.params['secc_cons_pc_rural'], m.tvalues['secc_cons_pc_rural']
    if base is None: base = c
    print(f'{lab:20s} {c:+8.4f} {t:7.2f} {c/base:14.0%}  {m.rsquared:.3f}  (n={len(Q)})')
print('\n  a channel that MEDIATES shrinks the coefficient toward 0.')
print('  if it survives EVERYTHING, prosperity works through something unmeasured.')

D.reset_index().to_parquet(f'{BASE}/mechanism.parquet', index=False)

=== IS IT BETA OR NOISE? ===
             beta  sys_sd  idio_sd  tot_sd  r2_sys
q                                                 
Q1 poorest  0.023   0.145    0.218   0.266   0.297
Q2          0.019   0.127    0.238   0.274   0.201
Q3          0.019   0.129    0.245   0.282   0.190
Q4          0.011   0.087    0.232   0.251   0.118
Q5 richest  0.009   0.073    0.231   0.244   0.080

  if beta and sys_sd FALL while idio_sd is FLAT -> prosperity cuts the cord, not adds noise
  beta     ~ prosperity (region FE): -0.0031 (t -2.98)
  sys_sd   ~ prosperity (region FE): -0.0145 (t -2.84)
  idio_sd  ~ prosperity (region FE): +0.0136 (t 1.07)

hort_share: mean 0.048, coverage 100%
  corr(hort_share, r2_sys)      = -0.198
  corr(hort_share, prosperity)  = +0.267

=== MEDIATION: does the prosperity effect survive each channel? ===
model                    coef       t   % of baseline  R2
baseline              -0.0477   -3.04           100%  0.511  (n=203)
+ horticulture        -0.0461   -2.78   

In [ ]:
# 05_factor — CELL 10: household-level diversification as the channel
import pandas as pd, numpy as np, statsmodels.api as sm
BASE='/content/drive/My Drive/Data/Elevar'
D = pd.read_parquet(f'{BASE}/mechanism.parquet').set_index('Dist Code')
SH = pd.read_parquet(f'{BASE}/shrug_features.parquet').set_index('dist_code')
for c in ['jobs_1d_p','secc_jobs_pc','land_own_share','secc_hh','nco2d_cultiv_share','hh_size','sc_sh','st_sh']:
    if c in SH.columns and c not in D.columns: D[c] = SH[c]
if 'jobs_1d_p' in D and 'secc_hh' in D: D['jobs_per_hh'] = D.jobs_1d_p/D.secc_hh

CAND = [c for c in ['jobs_per_hh','secc_jobs_pc','land_own_share','nco2d_cultiv_share','hh_size','sc_sh','st_sh'] if c in D.columns]
print('corr with r2_sys / beta / prosperity:')
for c in CAND:
    print(f'  {c:22s} {D[c].corr(D.r2_sys):+.3f}  {D[c].corr(D.beta):+.3f}  {D[c].corr(D.secc_cons_pc_rural):+.3f}')

base=None
print(f"\n{'model':26s} {'coef':>9s} {'t':>7s} {'%base':>7s}")
for lab, meds in [('baseline',[])] + [(f'+ {c}',[c]) for c in CAND] + [('+ ALL household',CAND)]:
    cols = ['secc_cons_pc_rural','literacy2']+meds
    Q = D.dropna(subset=cols+['r2_sys','region'])
    X = Q[cols].apply(lambda s:(s-s.mean())/s.std())
    X = pd.concat([X, pd.get_dummies(Q.region, prefix='r', drop_first=True, dtype=float)], axis=1)
    m = sm.OLS(Q.r2_sys.values, sm.add_constant(X)).fit(cov_type='cluster', cov_kwds={'groups':Q.region.values})
    c_, t_ = m.params['secc_cons_pc_rural'], m.tvalues['secc_cons_pc_rural']
    if base is None: base=c_
    print(f'{lab:26s} {c_:+9.4f} {t_:7.2f} {c_/base:6.0%}  (n={len(Q)})')

corr with r2_sys / beta / prosperity:
  jobs_per_hh            +0.012  +0.007  -0.048
  secc_jobs_pc           +0.013  +0.007  -0.050
  land_own_share         +0.122  +0.261  -0.108
  nco2d_cultiv_share     -0.031  -0.010  -0.148
  hh_size                +0.386  +0.284  -0.340
  sc_sh                  -0.208  -0.168  +0.337
  st_sh                  +0.230  +0.111  -0.451

model                           coef       t   %base
baseline                     -0.0477   -3.04   100%  (n=203)
+ jobs_per_hh                -0.0476   -2.94   100%  (n=203)
+ secc_jobs_pc               -0.0476   -2.94   100%  (n=203)
+ land_own_share             -0.0481   -3.11   101%  (n=203)
+ nco2d_cultiv_share         -0.0451   -2.88    95%  (n=171)
+ hh_size                    -0.0416   -2.66    87%  (n=203)
+ sc_sh                      -0.0430   -2.49    90%  (n=203)
+ st_sh                      -0.0507   -2.92   106%  (n=203)
+ ALL household              -0.0280   -1.63    59%  (n=171)


In [ ]:
# 05_factor — CELL 11: is the "ALL household" drop real, or sample + collinearity?
import pandas as pd, numpy as np, statsmodels.api as sm
BASE='/content/drive/My Drive/Data/Elevar'
D = pd.read_parquet(f'{BASE}/mechanism.parquet').set_index('Dist Code')
SH = pd.read_parquet(f'{BASE}/shrug_features.parquet').set_index('dist_code')
for c in ['jobs_1d_p','secc_hh','land_own_share','nco2d_cultiv_share','hh_size','sc_sh','st_sh']:
    if c in SH.columns and c not in D.columns: D[c] = SH[c]
D['jobs_per_hh'] = D.jobs_1d_p/D.secc_hh

def fit(cols, sub):
    Q = sub.dropna(subset=cols+['r2_sys','region'])
    X = Q[cols].apply(lambda s:(s-s.mean())/s.std())
    X = pd.concat([X, pd.get_dummies(Q.region, prefix='r', drop_first=True, dtype=float)], axis=1)
    m = sm.OLS(Q.r2_sys.values, sm.add_constant(X)).fit(cov_type='cluster', cov_kwds={'groups':Q.region.values})
    return m.params['secc_cons_pc_rural'], m.tvalues['secc_cons_pc_rural'], len(Q)

FULL = ['jobs_per_hh','land_own_share','nco2d_cultiv_share','hh_size','sc_sh','st_sh']
SUB  = D.dropna(subset=FULL+['r2_sys','region'])          # fix the sample at n=171
print(f'sample held at n={len(SUB)} throughout\n')
print(f"{'model':30s} {'coef':>9s} {'t':>7s} {'%base':>7s}")
b,t,n = fit(['secc_cons_pc_rural','literacy2'], SUB); base=b
print(f"{'baseline (n=171 sample)':30s} {b:+9.4f} {t:7.2f} {100:6.0f}%")
for c in FULL:
    b,t,n = fit(['secc_cons_pc_rural','literacy2',c], SUB)
    print(f"{'+ '+c:30s} {b:+9.4f} {t:7.2f} {b/base:6.0%}")
b,t,n = fit(['secc_cons_pc_rural','literacy2']+FULL, SUB)
print(f"{'+ ALL':30s} {b:+9.4f} {t:7.2f} {b/base:6.0%}")
# drop the development proxies, keep only genuinely economic channels
ECON = ['jobs_per_hh','land_own_share','nco2d_cultiv_share']
b,t,n = fit(['secc_cons_pc_rural','literacy2']+ECON, SUB)
print(f"{'+ economic channels only':30s} {b:+9.4f} {t:7.2f} {b/base:6.0%}")
DEMO = ['hh_size','sc_sh','st_sh']
b,t,n = fit(['secc_cons_pc_rural','literacy2']+DEMO, SUB)
print(f"{'+ demographic proxies only':30s} {b:+9.4f} {t:7.2f} {b/base:6.0%}")

print('\nVIF check (collinearity with prosperity):')
X = SUB[['secc_cons_pc_rural','literacy2']+FULL].apply(lambda s:(s-s.mean())/s.std())
for c in X.columns:
    r2 = sm.OLS(X[c].values, sm.add_constant(X.drop(columns=c).values)).fit().rsquared
    print(f'  {c:22s} VIF {1/(1-r2):5.1f}')

sample held at n=171 throughout

model                               coef       t   %base
baseline (n=171 sample)          -0.0444   -2.85    100%
+ jobs_per_hh                    -0.0441   -2.66    99%
+ land_own_share                 -0.0444   -2.90   100%
+ nco2d_cultiv_share             -0.0451   -2.88   102%
+ hh_size                        -0.0373   -2.29    84%
+ sc_sh                          -0.0385   -2.19    87%
+ st_sh                          -0.0455   -2.67   103%
+ ALL                            -0.0302   -1.62    68%
+ economic channels only         -0.0450   -2.74   102%
+ demographic proxies only       -0.0303   -1.61    68%

VIF check (collinearity with prosperity):
  secc_cons_pc_rural     VIF   1.8
  literacy2              VIF   1.5
  jobs_per_hh            VIF   1.3
  land_own_share         VIF   1.5
  nco2d_cultiv_share     VIF   1.4
  hh_size                VIF   1.9
  sc_sh                  VIF   1.8
  st_sh                  VIF   2.0
